## Example of a local alignment

In [1]:
import swalign

dna_string = "Homo sapeins"
reference_string = "Homo Sapience"
match_score = 2
mismatch_score = -1
matrix = swalign.NucleotideScoringMatrix(match_score, mismatch_score)
lalignment_object = swalign.LocalAlignment(matrix)
alignment_object = lalignment_object.align(dna_string, reference_string)

In [8]:
alignment_object.score

18

## Getting Family, Species and Subspecies from one pdf

In [2]:
import camelot
import pandas as pd

def read_table_in_pdf(filename, pages = 'all'):
    tables = camelot.read_pdf(filename, pages = pages)
    dfs = []
    
    for table in tables:
        df = table.df
        dfs.append(df)
    return dfs

def concat_dfs(dfs):
    df = pd.concat(dfs, ignore_index=True)
    return df

def promote_row_to_header(df, row):
    df.columns = list(df.iloc[row, :])
    return df

def remove_header_from_rows(df):
    count_header = sum(df[df.columns[0]] == df.columns[0])
    print(f'DEBUG: header appears {count_header} times as row.')
    df = df[df[df.columns[0]] != df.columns[0]]
    return df

def remove_row_by_col_val(df, val, colnum, exact = True):
    if exact:
        count_rows = sum(df[df.columns[colnum]] == val)
        print(f'DEBUG: pattern appears in {count_rows} rows.')
        df = df[df[df.columns[colnum]] != val]
    else:
        count_rows = sum(df[df.columns[colnum]].str.contains(val))
        print(f'DEBUG: pattern appears in {count_rows} rows.')
        df = df[~df[df.columns[colnum]].str.contains(val)]
    return df

def drop_na_col(df):
    return df.drop(df.columns[pd.isna(df.columns)], axis=1)

In [4]:
# Example 1: 
# 
dfs = read_table_in_pdf("../data/LI-2020-x-x-a-1-I.pdf")
data = concat_dfs(dfs)
data = promote_row_to_header(data, 0)
data = remove_header_from_rows(data)
data = drop_na_col(data)
data = remove_row_by_col_val(data, '', 0)
data = remove_row_by_col_val(data, 'Bitte bestellen Sie', 0, exact=False)
data = data.reset_index(drop=True)
# Malformatted row
row_idx = data[data[data.columns[0]].str.contains(' ')].index.tolist()
# separate
values = data.iloc[row_idx[0]][data.columns[0]].split()
data.iloc[row_idx[0]][data.columns[0]] = values[0]
data.iloc[row_idx[0]][data.columns[1]] = values[1]

DEBUG: header appears 20 times as row.
DEBUG: pattern appears in 9 rows.
DEBUG: pattern appears in 1 rows.


In [22]:
tmp = data.apply(lambda x: ','.join(data['Familie'] +' '+ data['Gattung'])+' '+data['Species \n(Subspecies, \nVarietät)'])

In [29]:
names = tmp['Familie'][0].split(',')
print(names)

['Acanthaceae Ruellia', 'Aceraceae Acer', 'Aizoaceae Delosperma', 'Aizoaceae Delosperma', 'Aizoaceae Delosperma', 'Aizoaceae Delosperma', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Allium', 'Alliaceae Nectaroscordum', 'Annonaceae Asimina', 'Anthericaceae Anthericum', 'Anthericaceae Anthericum', 'Apiaceae Eryngium', 'Apiaceae Eryngium', 'Apiaceae Meum', 'Apiaceae Orlaya', 'Apiaceae Seseli', 'Araceae Arum', 'Araliaceae Aralia', 'Aristolochiaceae Aristolochia', 'Asparagaceae Bellevalia', 'Asparagaceae Triteleia', 'Asphodelaceae Asphodeline', 'Asphodelaceae Asphodelus', 'Asphodelaceae Paradisea', 'Asteraceae Achillea', 'Asteraceae Achillea', 'Asteraceae Anacyclus', 'Asteraceae Arnica', 'Asteraceae Arnica', 'Asteraceae Artemisia', 'Asteraceae Artemisia', 'Asteraceae Aster', 'Asteraceae Aster', 'Asteraceae Aster', 'Asteraceae Aster', 'Asteraceae Aster', 'Asteraceae Buphthalmum', 'Aster

## Calculating alignment scores between all extracted names

In [48]:
from itertools import product
scores = {}
for n1, n2 in list(product(names, names)):
    if n1[0] == n2[0]:
        scores[(n1, n2)] = lalignment_object.align(n1, n2).score/len(n1)

In [49]:
scores

{('Acanthaceae Ruellia', 'Acanthaceae Ruellia'): 2.0,
 ('Acanthaceae Ruellia', 'Aceraceae Acer'): 0.631578947368421,
 ('Acanthaceae Ruellia', 'Aizoaceae Delosperma'): 0.7368421052631579,
 ('Acanthaceae Ruellia', 'Alliaceae Allium'): 0.7894736842105263,
 ('Acanthaceae Ruellia', 'Alliaceae Nectaroscordum'): 0.631578947368421,
 ('Acanthaceae Ruellia', 'Annonaceae Asimina'): 0.6842105263157895,
 ('Acanthaceae Ruellia', 'Anthericaceae Anthericum'): 0.8421052631578947,
 ('Acanthaceae Ruellia', 'Apiaceae Eryngium'): 0.6842105263157895,
 ('Acanthaceae Ruellia', 'Apiaceae Meum'): 0.631578947368421,
 ('Acanthaceae Ruellia', 'Apiaceae Orlaya'): 0.6842105263157895,
 ('Acanthaceae Ruellia', 'Apiaceae Seseli'): 0.7368421052631579,
 ('Acanthaceae Ruellia', 'Araceae Arum'): 0.7894736842105263,
 ('Acanthaceae Ruellia', 'Araliaceae Aralia'): 0.8421052631578947,
 ('Acanthaceae Ruellia', 'Aristolochiaceae Aristolochia'): 0.7368421052631579,
 ('Acanthaceae Ruellia', 'Asparagaceae Bellevalia'): 0.8421052631

A score of `2` indicates perfect match.